# ENLIGHT Account Validation - Missing CA Finder

Compare program CSV export with validation rules to find missing CAs.

In [ ]:
import pandas as pd
import os
from collections import defaultdict

## 1. Load Program CSV

Update `program_csv_path` to point to your program's CSV export file.

In [ ]:
# Update this path to your program CSV file
program_csv_path = "/tmp/CA_BP_rules.csv"  # Example path

if os.path.exists(program_csv_path):
    df = pd.read_csv(program_csv_path, header=None, names=['Rule', 'Value'])
    print(f"Loaded {len(df)} records")
    print(f"\nFirst 10 rows:")
    display(df.head(10))
else:
    print(f"File not found: {program_csv_path}")
    print("\nPlease update the path to your program CSV file.")

## 2. Group by Rule

Shows count per rule in the program export.

In [ ]:
rule_summary = df.groupby('Rule').size().reset_index(name='Count')
rule_summary = rule_summary.sort_values('Count', ascending=False)
print("Program CSV - Count by Rule:")
display(rule_summary)

## 3. Find Missing CA

Enter the CA number you want to check. The notebook will show which rules contain it and if it's missing.

In [ ]:
# Enter CA to search
ca_to_find = "1234567890"  # <-- Change this to the CA number you're checking

print(f"Searching for CA: {ca_to_find}")
print("=" * 50)

found_rules = []
for rule in df['Rule'].unique():
    rule_data = df[df['Rule'] == rule]
    matches = rule_data[rule_data['Value'] == ca_to_find]
    if len(matches) > 0:
        found_rules.append(rule)
        print(f"✅ Found in {rule}")

if not found_rules:
    print(f"❌ NOT FOUND in any rule!")
    print("\nThis CA is missing from the program export.")

## 4. Compare Validation vs Program

Load validation data and compare counts.

In [ ]:
# Enter validation count (from ABAP report)
validation_counts = {
    'Rule 1 - Active CA': 0,  # <-- Enter from ABAP report
    'Rule 2 - Inactive CA': 0,
    'Rule 3 - Write-off CA': 0,
    'Rule 4 - Outstanding CA': 0,
    'Rule 9 - Shutdown CA': 0,
}

print("Compare Validation Count vs Program Export:")
print("=" * 60)
print(f"{'Rule':<30} {'Validation':>12} {'Program':>12} {'Diff':>10}")
print("-" * 60)

for rule_name, val_count in validation_counts.items():
    # Map to program rule
    if 'Rule1' in rule_name:
        prog_rule = 'Rule1: Vkont'
    elif 'Rule2' in rule_name:
        prog_rule = 'Rule2: Vkont'
    elif 'Rule3' in rule_name:
        prog_rule = 'Rule3: Vkont'
    elif 'Rule4' in rule_name:
        prog_rule = 'Rule4: Vkont'
    elif 'Rule9' in rule_name:
        prog_rule = 'Rule9: Vkont'
    else:
        prog_rule = None
    
    prog_count = len(df[df['Rule'] == prog_rule]) if prog_rule else 0
    diff = val_count - prog_count
    diff_str = f"{diff:+" if diff >= 0 else str(diff) 
    
    status = "✅" if diff == 0 else "⚠️"
    print(f"{rule_name:<30} {val_count:>12} {prog_count:>12} {diff_str:>10} {status}")

## 5. Find All Missing CAs (Validation - Program)

This compares the full list. Requires validation export.

In [ ]:
# Load validation export (if available)
validation_csv_path = "/tmp/validation_ca.csv"  # <-- Update if you have validation export

if os.path.exists(validation_csv_path):
    df_validation = pd.read_csv(validation_csv_path, header=None, names=['VKONT'])
    df_program = df[df['Rule'].str.contains('Vkont', na=False)][['Value']].drop_duplicates()
    df_program.columns = ['VKONT']
    
    missing = set(df_validation['VKONT']) - set(df_program['VKONT'])
    
    print(f"Validation CA count: {len(df_validation)}")
    print(f"Program CA count: {len(df_program)}")
    print(f"Missing in program: {len(missing)}")
    
    if len(missing) > 0:
        print(f"\nMissing CAs:")
        for ca in sorted(missing)[:20]:  # Show first 20
            print(f"  {ca}")
        if len(missing) > 20:
            print(f"  ... and {len(missing) - 20} more")
else:
    print("Validation export not found. Please export from ABAP report first.")

---

## How to Use

1. **Run cell 1** - Load program CSV
2. **Update `program_csv_path`** with your file path
3. **Run cells 2-4** - View summary and search for specific CA
4. **Enter validation counts** in cell 4 for comparison
5. **Export validation CSV** to compare full lists in cell 5